# 🎬 Hệ Thống Gợi Ý Phim Hybrid AI (MovieLens 32M)
### Kết hợp Collaborative Filtering (CF) & Machine Learning (LightGBM Ranker)

**Quy mô dữ liệu:**
- **32,000,204** lượt đánh giá (ratings) từ **200,948** người dùng trên **87,585** bộ phim.
- Hơn **2,000,000** lượt gán thẻ nội dung (tags).

**Kiến trúc 2 vòng chuẩn công nghiệp (Two-Stage Recommendation Pipeline):**
1. **Giai đoạn 1 (Phase 1):** Chuẩn bị & Chế biến dữ liệu (Data Preprocessing & Feature Engineering).
2. **Giai đoạn 2 (Phase 2):** Lọc nhanh ứng viên (Candidate Retrieval via Collaborative Filtering - Top 300).
3. **Giai đoạn 3 (Phase 3):** Chấm điểm & Xếp hạng chi tiết (Scoring & Re-ranking via LightGBM Classifier/Ranker).
4. **Giai đoạn 4 (Phase 4):** Pipeline gợi ý hoàn chỉnh & Đánh giá kết quả (Top 10 Recommendation Demo).

## 🛠️ Bước 0: Cài đặt thư viện & Chuẩn bị môi trường

In [ ]:
# Cài đặt các thư viện cần thiết trên Colab nếu chưa có
!pip install -q lightgbm scikit-learn pandas numpy scipy

import os
import re
import zipfile
import urllib.request
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = './data'
ZIP_PATH = './ml-32m.zip'
os.makedirs(DATA_DIR, exist_ok=True)

# Tự động tải dữ liệu MovieLens 32M nếu chạy trên môi trường mới
if not os.path.exists(ZIP_PATH):
    print('Đang tải MovieLens 32M (238 MB)...')
    url = 'https://files.grouplens.org/datasets/movielens/ml-32m.zip'
    urllib.request.urlretrieve(url, ZIP_PATH)
    print('✅ Tải dữ liệu hoàn tất!')
else:
    print(f'✅ Đã tìm thấy tệp dữ liệu: {ZIP_PATH}')

✅ Đã tìm thấy tệp dữ liệu: ./ml-32m.zip


## 🚀 GIAI ĐOẠN 1: Chuẩn Bị & Chế Biến Dữ Liệu (Feature Engineering)

Giai đoạn này giải quyết 4 nhiệm vụ cốt lõi:
1. **Xử lý phim (`movies.csv`):** Tách năm phát hành (`release_year`), tính tuổi phim (`movie_age`), và mã hóa 18 thể loại (`genre_*`).
2. **Khai phá thẻ (`tags.csv`):** Sử dụng thuật toán TF-IDF trích xuất 30 đặc trưng phong cách nội dung (twist ending, atmospheric, sci-fi...).
3. **Thống kê toàn cục (`ratings.csv`):** Quét toàn bộ 32M dòng bằng kỹ thuật chunking để đo độ phổ biến và điểm trung bình.
4. **Gán nhãn giám sát & Chia tập theo thời gian:** $y = 1$ (Thích nếu rating $\ge 4.0$) và chia 80% quá khứ để Train, 20% gần đây để Test.

### 1.1 & 1.2: Xử lý thông tin phim & Mã hóa thể loại, tags

In [ ]:
print('1. Đang đọc và làm sạch movies.csv...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open('ml-32m/movies.csv') as f:
        df_movies = pd.read_csv(f)

print(f'Tổng số phim ban đầu: {len(df_movies):,}')

1. Đang đọc và làm sạch movies.csv...
Tổng số phim ban đầu: 87,585


In [ ]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Thêm feature year (tách từ feature title ra) : có 2 cách
- 1 là viêt function extract_year(title)
- 2 là viết trên cùng một hàm -> cách này sẽ tối ưu tốc độ trên Pandas

In [ ]:
# cách 1
# Trích xuất năm phát hành từ title
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else np.nan

df_movies['release_year'] = df_movies['title'].apply(extract_year)


df_movies.head()

,movieId,title,genres,release_year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0


In [ ]:
# cách 2
df_movies['release_year'] = df_movies['title'].str.extract(r'\((\d{4})\)').astype(float)

df_movies.head()

,movieId,title,genres,release_year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0


kiểm tra xem là trong release_year nó có các ô bị nan hay không , nếu có thì đi chuẩn hóa

In [ ]:
df_movies[df_movies["release_year"].isna()]

,movieId,title,genres,release_year
15038,79607,"Millions Game, The (Das Millionenspiel)",Action|Drama|Sci-Fi|Thriller,NaN
25425,123619,Terrible Joe Moran,(no genres listed),NaN
26356,125571,The Court-Martial of Jackie Robinson,(no genres listed),NaN
26381,125632,In Our Garden,(no genres listed),NaN
26466,125958,Stephen Fry In America - New World,(no genres listed),NaN
...,...,...,...,...
86528,288927,Fortunes,Comedy,NaN
86634,289193,The Wonderful Story of Henry Sugar,Adventure|Children|Comedy,NaN
86833,289909,Rattler,(no genres listed),NaN
87074,290775,Being Romanian: A Family Journal,(no genres listed),NaN


Chuẩn hóa lại cột release_year
- bước 1 là tính năm trung bình có trong tất cả các year dùng medican để tính
- bước 2  là những ô nào trong release_year mà bị NaN thì lúc này sẽ thay thành cái năm trung bình đó

In [ ]:

median_year = df_movies['release_year'].median()
print(median_year)
df_movies['release_year'] = df_movies['release_year'].fillna(median_year).astype(int)

2006.0


kiểm tra lại là đã chuẩn hóa cột release_year hay chưa

In [ ]:
df_movies[df_movies["release_year"].isna()]

,movieId,title,genres,release_year


In [ ]:
df_movies.head()

,movieId,title,genres,release_year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II (1995),Comedy,1995


Thêm 2 feature là movie_age và clean_title
- movie_age là tính tuổi đời của bộ phim đến năm hiện tại là 2026
- clean_title là cột title mà nó sẽ không có year ở trong đó

In [ ]:
df_movies['movie_age'] = 2026 - df_movies['release_year']
df_movies['clean_title'] = df_movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True).str.strip()

df_movies.head()

,movieId,title,genres,release_year,movie_age,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,31,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,31,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,31,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,31,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,1995,31,Father of the Bride Part II


## Bước tiếp theo là multi-hot enconding
- mục đích của cái việc này là sẽ tạo ra các feature riêng cho từng thể loại , nếu phim có thể loại nào thì feature thể loại đó sẽ đánh dấu là 1 , còn không có thì là 0


Kiểm tra bảng genres hiện tại đang có những thể loại nào

In [ ]:
genre_counts = df_movies['genres'].str.split('|').explode().value_counts()
print(genre_counts)

genres
Drama                 34175
Comedy                23124
Thriller              11823
Romance               10369
Action                 9668
Documentary            9363
Horror                 8654
(no genres listed)     7080
Crime                  6976
Adventure              5402
Sci-Fi                 4907
Animation              4617
Children               4520
Mystery                4013
Fantasy                3851
War                    2325
Western                1696
Musical                1059
Film-Noir               353
IMAX                    195
Name: count, dtype: int64


Lấy danh sách các thể loại không bao gồm mấy cái không phải là genres như là (no genres listed) và IMAX

In [ ]:
all_genres = sorted(
    set(g for genres in df_movies['genres'].str.split('|') for g in genres)
    - {'(no genres listed)', 'IMAX'}
)
print(all_genres)
print(len(all_genres))

['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
18


Thêm fearture cho từng genre

In [ ]:
for g in all_genres:
    pattern = r'Children' if g == 'Children' else g
    df_movies[f'genre_{g}'] = df_movies['genres'].str.contains(pattern, regex=True).astype(int)

df_movies.head()

,movieId,title,genres,release_year,movie_age,clean_title,genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,31,Toy Story,0,1,1,1,...,1,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,31,Jumanji,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,31,Grumpier Old Men,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,31,Waiting to Exhale,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995,31,Father of the Bride Part II,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Tiếp theo là thêm cột feautre genre_count để show ra một bộ phim thì nó có tổng bao nhiêu là genres

In [ ]:
df_movies['genre_count'] = df_movies[[f'genre_{g}' for g in all_genres]].sum(axis=1)
df_movies.head()

,movieId,title,genres,release_year,movie_age,clean_title,genre_Action,genre_Adventure,genre_Animation,genre_Children,...,genre_Film-Noir,genre_Horror,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,genre_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,31,Toy Story,0,1,1,1,...,0,0,0,0,0,0,0,0,0,5
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,31,Jumanji,0,1,0,1,...,0,0,0,0,0,0,0,0,0,3
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,31,Grumpier Old Men,0,0,0,0,...,0,0,0,0,1,0,0,0,0,2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,31,Waiting to Exhale,0,0,0,0,...,0,0,0,0,1,0,0,0,0,3
4,5,Father of the Bride Part II (1995),Comedy,1995,31,Father of the Bride Part II,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [ ]:
df_movies[['movieId', 'title', 'genre_count']].head(10)

,movieId,title,genre_count
0,1,Toy Story (1995),5
1,2,Jumanji (1995),3
2,3,Grumpier Old Men (1995),2
3,4,Waiting to Exhale (1995),3
4,5,Father of the Bride Part II (1995),1
5,6,Heat (1995),3
6,7,Sabrina (1995),2
7,8,Tom and Huck (1995),2
8,9,Sudden Death (1995),1
9,10,GoldenEye (1995),3


In [ ]:
print('2. Đang tổng hợp tags.csv bằng TF-IDF...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open('ml-32m/tags.csv') as f:
        df_tags = pd.read_csv(f, usecols=['movieId', 'tag'])

df_tags.head()

2. Đang tổng hợp tags.csv bằng TF-IDF...


,movieId,tag
0,26479,Kevin Kline
1,79592,misogyny
2,247150,acrophobia
3,2174,music
4,2174,weird


In [ ]:
df_tags.value_counts()

,,count
movieId,tag,
260,sci-fi,596
79132,alternate reality,360
296,Quentin Tarantino,355
2959,twist ending,347
79132,thought-provoking,339
...,...,...
292143,village,1
292349,politically incorrect,1
292371,Stephen King,1


In [ ]:
df_tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000072 entries, 0 to 2000071
Data columns (total 2 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   movieId  int64 
 1   tag      object
dtypes: int64(1), object(1)
memory usage: 30.5+ MB


Kiểm tra xem là có cái nào bị nan hay không

In [ ]:
df_tags.isna().sum()

,0
movieId,0
tag,17


xóa bỏ những ô nào của feature tag mà bị nan

In [ ]:
df_tags = df_tags.dropna(subset=['tag'])
df_tags.isna().sum()

,0
movieId,0
tag,0


Chuẩn hóa lại định các ô của feature tag , đó là chuyển tất cả thành chuỗi, thành chữ thường, và loại bỏ khoản trắng ở 2 đầu

In [ ]:
df_tags['tag'] = df_tags['tag'].astype(str).str.lower().str.strip()

df_tags.head()

,movieId,tag
0,26479,kevin kline
1,79592,misogyny
2,247150,acrophobia
3,2174,music
4,2174,weird


Gôm các thẻ tag của từng bộ phim riêng lẻ lại thành một string duy nhất cho từng bộ phim và chúng sẽ cách nhau bằng dấu cách


Ở đây chỉ nối chúng lại với nhau cho phép lặp lại để có thể làm "tần suất lặp lại" để đo độ quan trọng

In [ ]:
movie_tags = df_tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x.astype(str))).reset_index()

movie_tags.head(20)

,movieId,tag
0,1,children disney animation children disney disn...
1,2,robin williams fantasy robin williams time tra...
2,3,comedinha de velhinhos engraãƒâ§ada comedinha ...
3,4,characters slurs based on novel or book chick ...
4,5,fantasy pregnancy remake family steve martin s...
5,6,atmospheric dialogue tumey's dvds atmospheric ...
6,7,as good maybe better than original great cast ...
7,8,library system based on a book mark twain 19th...
8,9,action jean-claude van damme can't remember ac...
9,10,james bond tumey's dvds 007 bond casual sex es...


lấy ra 30 từ khóa mà xuất hiện nhiều nhất và loại bỏ các từ nối tiếng anh như là" the, a,an, of,and,..."
sau đó tạo một list chứa tên các feature của từng tag

In [ ]:
# Trích xuất top 30 từ khóa đặc trưng nhất
tfidf = TfidfVectorizer(max_features=30, stop_words='english')
tfidf_matrix = tfidf.fit_transform(movie_tags['tag'])
tag_feature_names = [f'tag_{re.sub(r"[^a-zA-Z0-9_]", "_", feat)}' for feat in tfidf.get_feature_names_out()]
print(tfidf_matrix)
print("================")
print(tag_feature_names)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 88802 stored elements and shape (51323, 30)>
  Coords	Values
  (0, 13)	0.877069633274224
  (0, 6)	0.3774111165088366
  (0, 18)	0.23723150054510783
  (0, 14)	0.08404327826289533
  (0, 24)	0.020333209089741473
  (0, 11)	0.0202650883121869
  (0, 25)	0.04812892820239005
  (0, 12)	0.07209333527828406
  (0, 0)	0.0708550478146968
  (0, 5)	0.09454847973160642
  (0, 23)	0.025886049418887922
  (0, 10)	0.014504367029637883
  (0, 22)	0.034201755771450154
  (0, 9)	0.016749436591945367
  (0, 15)	0.018413945048298797
  (1, 6)	0.21870312133037004
  (1, 12)	0.09399785198760564
  (1, 0)	0.13857499319496977
  (1, 5)	0.18491350072428372
  (1, 23)	0.303760358594211
  (1, 10)	0.05673392720546608
  (1, 22)	0.5351215722193502
  (1, 9)	0.06551553159115849
  (1, 2)	0.35492499622787704
  (1, 4)	0.4659266978632246
  :	:
  (51297, 11)	0.3020227599391948
  (51297, 21)	0.5232473751466628
  (51297, 28)	0.3497530176458524
  (51297, 27)	0.6487181446673496
  

Thêm dataframe cho df_tag_features : nó sẽ chứa mức độ nhận dạng của mỗi phim tương ứng với mỗi tag từ 30 tag ở trên, càng về 1 thì càng cao

In [ ]:
df_tag_features = pd.DataFrame(tfidf_matrix.toarray(), columns=tag_feature_names)

df_tag_features.head()

,tag_action,tag_atmospheric,tag_based,tag_black,tag_book,tag_character,tag_comedy,tag_dark,tag_death,tag_ending,...,tag_nudity,tag_plot,tag_reference,tag_relationship,tag_sci,tag_story,tag_twist,tag_visually,tag_war,tag_woman
0,0.070855,0.0,0.000000,0.00000,0.000000,0.094548,0.377411,0.0,0.0,0.016749,...,0.0,0.0,0.034202,0.025886,0.020333,0.048129,0.0,0.0,0.0,0.000000
1,0.138575,0.0,0.354925,0.06395,0.465927,0.184914,0.218703,0.0,0.0,0.065516,...,0.0,0.0,0.535122,0.303760,0.000000,0.000000,0.0,0.0,0.0,0.148351
2,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000
3,0.000000,0.0,0.549121,0.00000,0.630750,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.548289,0.000000,0.000000,0.0,0.0,0.0,0.000000
4,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.475142,0.0,0.0,0.000000,...,0.0,0.0,0.000000,0.879909,0.000000,0.000000,0.0,0.0,0.0,0.000000


In [ ]:
df_tag_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51323 entries, 0 to 51322
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   tag_action        51323 non-null  float64
 1   tag_atmospheric   51323 non-null  float64
 2   tag_based         51323 non-null  float64
 3   tag_black         51323 non-null  float64
 4   tag_book          51323 non-null  float64
 5   tag_character     51323 non-null  float64
 6   tag_comedy        51323 non-null  float64
 7   tag_dark          51323 non-null  float64
 8   tag_death         51323 non-null  float64
 9   tag_ending        51323 non-null  float64
 10  tag_female        51323 non-null  float64
 11  tag_fi            51323 non-null  float64
 12  tag_film          51323 non-null  float64
 13  tag_funny         51323 non-null  float64
 14  tag_good          51323 non-null  float64
 15  tag_great         51323 non-null  float64
 16  tag_horror        51323 non-null  float6

Thêm cột movieId tương ứng vào df_tag_features

In [ ]:
df_tag_features['movieId'] = movie_tags['movieId']
df_tag_features.head()

,tag_action,tag_atmospheric,tag_based,tag_black,tag_book,tag_character,tag_comedy,tag_dark,tag_death,tag_ending,...,tag_plot,tag_reference,tag_relationship,tag_sci,tag_story,tag_twist,tag_visually,tag_war,tag_woman,movieId
0,0.070855,0.0,0.000000,0.00000,0.000000,0.094548,0.377411,0.0,0.0,0.016749,...,0.0,0.034202,0.025886,0.020333,0.048129,0.0,0.0,0.0,0.000000,1
1,0.138575,0.0,0.354925,0.06395,0.465927,0.184914,0.218703,0.0,0.0,0.065516,...,0.0,0.535122,0.303760,0.000000,0.000000,0.0,0.0,0.0,0.148351,2
2,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,3
3,0.000000,0.0,0.549121,0.00000,0.630750,0.000000,0.000000,0.0,0.0,0.000000,...,0.0,0.000000,0.548289,0.000000,0.000000,0.0,0.0,0.0,0.000000,4
4,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.475142,0.0,0.0,0.000000,...,0.0,0.000000,0.879909,0.000000,0.000000,0.0,0.0,0.0,0.000000,5


In [ ]:
df_tag_features.shape

(51323, 31)

In [ ]:
df_movies.shape

(87585, 25)

Đóng gói toàn bộ thông tin từ movies và tag vào thành một là df_movies_final

In [ ]:
# Ghép các đặc trưng phim
df_movies_final = df_movies.merge(df_tag_features, on='movieId', how='left')

df_movies_final.head()

,movieId,title,genres,release_year,movie_age,clean_title,genre_Action,genre_Adventure,genre_Animation,genre_Children,...,tag_nudity,tag_plot,tag_reference,tag_relationship,tag_sci,tag_story,tag_twist,tag_visually,tag_war,tag_woman
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,31,Toy Story,0,1,1,1,...,0.0,0.0,0.034202,0.025886,0.020333,0.048129,0.0,0.0,0.0,0.000000
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,31,Jumanji,0,1,0,1,...,0.0,0.0,0.535122,0.303760,0.000000,0.000000,0.0,0.0,0.0,0.148351
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,31,Grumpier Old Men,0,0,0,0,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,31,Waiting to Exhale,0,0,0,0,...,0.0,0.0,0.000000,0.548289,0.000000,0.000000,0.0,0.0,0.0,0.000000
4,5,Father of the Bride Part II (1995),Comedy,1995,31,Father of the Bride Part II,0,0,0,0,...,0.0,0.0,0.000000,0.879909,0.000000,0.000000,0.0,0.0,0.0,0.000000


In [ ]:
df_movies_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 55 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movieId            87585 non-null  int64  
 1   title              87585 non-null  object 
 2   genres             87585 non-null  object 
 3   release_year       87585 non-null  int64  
 4   movie_age          87585 non-null  int64  
 5   clean_title        87585 non-null  object 
 6   genre_Action       87585 non-null  int64  
 7   genre_Adventure    87585 non-null  int64  
 8   genre_Animation    87585 non-null  int64  
 9   genre_Children     87585 non-null  int64  
 10  genre_Comedy       87585 non-null  int64  
 11  genre_Crime        87585 non-null  int64  
 12  genre_Documentary  87585 non-null  int64  
 13  genre_Drama        87585 non-null  int64  
 14  genre_Fantasy      87585 non-null  int64  
 15  genre_Film-Noir    87585 non-null  int64  
 16  genre_Horror       875

Chuẩn hóa lại một số các tag của các phim bị thiếu

In [ ]:
for col in tag_feature_names:
    df_movies_final[col] = df_movies_final[col].fillna(0.0)

df_movies_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 55 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movieId            87585 non-null  int64  
 1   title              87585 non-null  object 
 2   genres             87585 non-null  object 
 3   release_year       87585 non-null  int64  
 4   movie_age          87585 non-null  int64  
 5   clean_title        87585 non-null  object 
 6   genre_Action       87585 non-null  int64  
 7   genre_Adventure    87585 non-null  int64  
 8   genre_Animation    87585 non-null  int64  
 9   genre_Children     87585 non-null  int64  
 10  genre_Comedy       87585 non-null  int64  
 11  genre_Crime        87585 non-null  int64  
 12  genre_Documentary  87585 non-null  int64  
 13  genre_Drama        87585 non-null  int64  
 14  genre_Fantasy      87585 non-null  int64  
 15  genre_Film-Noir    87585 non-null  int64  
 16  genre_Horror       875

In [ ]:
df_movies_final.head(20)

,movieId,title,genres,release_year,movie_age,clean_title,genre_Action,genre_Adventure,genre_Animation,genre_Children,...,tag_nudity,tag_plot,tag_reference,tag_relationship,tag_sci,tag_story,tag_twist,tag_visually,tag_war,tag_woman
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,31,Toy Story,0,1,1,1,...,0.000000,0.000000,0.034202,0.025886,0.020333,0.048129,0.0,0.0,0.000000,0.000000
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,31,Jumanji,0,1,0,1,...,0.000000,0.000000,0.535122,0.303760,0.000000,0.000000,0.0,0.0,0.000000,0.148351
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,31,Grumpier Old Men,0,0,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,31,Waiting to Exhale,0,0,0,0,...,0.000000,0.000000,0.000000,0.548289,0.000000,0.000000,0.0,0.0,0.000000,0.000000
4,5,Father of the Bride Part II (1995),Comedy,1995,31,Father of the Bride Part II,0,0,0,0,...,0.000000,0.000000,0.000000,0.879909,0.000000,0.000000,0.0,0.0,0.000000,0.000000
5,6,Heat (1995),Action|Crime|Thriller,1995,31,Heat,1,0,0,0,...,0.000000,0.000000,0.000000,0.017883,0.000000,0.000000,0.0,0.0,0.000000,0.000000
6,7,Sabrina (1995),Comedy|Romance,1995,31,Sabrina,0,0,0,0,...,0.000000,0.000000,0.105607,0.559511,0.000000,0.594444,0.0,0.0,0.000000,0.000000
7,8,Tom and Huck (1995),Adventure|Children,1995,31,Tom and Huck,0,1,0,1,...,0.000000,0.000000,0.000000,0.263940,0.000000,0.000000,0.0,0.0,0.000000,0.000000
8,9,Sudden Death (1995),Action,1995,31,Sudden Death,1,0,0,0,...,0.093435,0.111325,0.000000,0.492484,0.000000,0.000000,0.0,0.0,0.000000,0.160346
9,10,GoldenEye (1995),Action|Adventure|Thriller,1995,31,GoldenEye,1,1,0,0,...,0.000000,0.141862,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.306496


### 1.3 & 1.4: Thống kê toàn cục trên 32M ratings & Chia tập Train/Test theo thời gian

đọcfile rating.csv, mỗi lần đọc thì đọc 2 triệu dòng , mỗi lần đọc sẽ cộng dồn điểm
- movie_sum sẽ tính là mỗi bộ phim sẽ cỏ tổng số điểm đánh giá là bao nhiêu
- movie_coun sẽ tính là có bao nhiêu lượt đáng giá cho bộ phim
- user_sum là tổng số điểm mà user đã đánh giá
- user_count là số lượt user đánh giá  

In [ ]:
chunk_size = 2_000_000
movie_sum = {}
movie_count = {}
user_sum = {}
user_count = {}
total_rows = 0

In [ ]:
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open('ml-32m/ratings.csv') as f:
        reader = pd.read_csv(f, chunksize=chunk_size, dtype={'userId': np.int32, 'movieId': np.int32, 'rating': np.float32, 'timestamp': np.int64})
        for chunk in reader:
            total_rows += len(chunk)
            # Movie stats
            m_grp = chunk.groupby('movieId')['rating'].agg(['sum', 'count'])
            for m_id, row in m_grp.iterrows():
                movie_sum[m_id] = movie_sum.get(m_id, 0.0) + row['sum']
                movie_count[m_id] = movie_count.get(m_id, 0) + int(row['count'])
            # User stats
            u_grp = chunk.groupby('userId')['rating'].agg(['sum', 'count'])
            for u_id, row in u_grp.iterrows():
                user_sum[u_id] = user_sum.get(u_id, 0.0) + row['sum']
                user_count[u_id] = user_count.get(u_id, 0) + int(row['count'])

Tạo bảng df_movie_status , để xem các giá trị tb của một bộ phim

In [ ]:
df_movie_stats = pd.DataFrame({
    'movieId': list(movie_count.keys()),
    'movie_rating_count': list(movie_count.values()),
    'movie_avg_rating': [movie_sum[m] / movie_count[m] for m in movie_count.keys()]
})
df_movie_stats.head()

,movieId,movie_rating_count,movie_avg_rating
0,1,68997,3.897438
1,2,28904,3.275758
2,3,13134,3.139447
3,4,2806,2.845331
4,5,13154,3.059602


Thêm feature movie_log_count cho df_movie_stats để thay thế cho movie_avg_rating để chuẩn hóa thang đo độ phổ biến, giúp các thuật toán máy học xử lý công bằng hơn tránh kết quả sẽ ưu tiên các movie_avg_rating cao quá mức

Ví dụ như : một phim có 20 nghìn lượt xem và một phim có 100 nghìn người xem và điểm trung bình giữa 2 bộ phim bị chênh lệch quá lớn thì lúc nào hệ thống cũng sẽ gợi ý phim 100 nghìn nười xem mặt dù là cả 2 cùng nhiều lượt xem để khắc phục thì giảm bớt sự chênh lệch đó đi bằng cách là log nó

In [ ]:
df_movie_stats['movie_log_count'] = np.log1p(df_movie_stats['movie_rating_count'])

df_movie_stats.head()

,movieId,movie_rating_count,movie_avg_rating,movie_log_count
0,1,68997,3.897438,11.141833
1,2,28904,3.275758,10.271770
2,3,13134,3.139447,9.483036
3,4,2806,2.845331,7.939872
4,5,13154,3.059602,9.484557


Tạo df_user_stats để xem user thường đánh giá bao nhiều chấm bao nhiêu cho một bộ phim

In [ ]:
df_user_stats = pd.DataFrame({
    'userId': list(user_count.keys()),
    'user_rating_count': list(user_count.values()),
    'user_avg_rating': [user_sum[u] / user_count[u] for u in user_count.keys()]
})
df_user_stats['user_log_count'] = np.log1p(df_user_stats['user_rating_count'])

df_movie_stats.head()

,movieId,movie_rating_count,movie_avg_rating,movie_log_count
0,1,68997,3.897438,11.141833
1,2,28904,3.275758,10.271770
2,3,13134,3.139447,9.483036
3,4,2806,2.845331,7.939872
4,5,13154,3.059602,9.484557


tổng số lượng user

In [ ]:
len(df_user_stats)

200948

Lấy ra tối đá 20 nghìn người dùng tích cực đánh giá nhất (với điều kiện là mỗi người được chọn phải có ít nhất 30 lượt đánh giá) để làm giữ liệu train tốt

In [ ]:
print('4. Lấy mẫu 20,000 users tích cực và chia tập Train/Test theo thời gian...')
top_users = df_user_stats[df_user_stats['user_rating_count'] >= 30].nlargest(20000, 'user_rating_count')['userId'].values
print(top_users)
print(len(top_users))
top_users_set=set(top_users)
print(top_users_set)
print(len(top_users_set))

4. Lấy mẫu 20,000 users tích cực và chia tập Train/Test theo thời gian...
[175325  17035  55653 ... 102438 102687 102838]
20000
{np.int64(131079), np.int64(10), np.int64(131083), np.int64(28), np.int64(131100), np.int64(131102), np.int64(35), np.int64(131116), np.int64(131119), np.int64(131122), np.int64(131131), np.int64(59), np.int64(65), np.int64(70), np.int64(131143), np.int64(131144), np.int64(131145), np.int64(76), np.int64(87), np.int64(131162), np.int64(131178), np.int64(109), np.int64(115), np.int64(128), np.int64(138), np.int64(131216), np.int64(151), np.int64(157), np.int64(158), np.int64(163), np.int64(131249), np.int64(187), np.int64(188), np.int64(131262), np.int64(131279), np.int64(208), np.int64(131281), np.int64(131286), np.int64(131294), np.int64(228), np.int64(131300), np.int64(238), np.int64(239), np.int64(240), np.int64(242), np.int64(245), np.int64(131325), np.int64(131334), np.int64(265), np.int64(268), np.int64(283), np.int64(131355), np.int64(311), np.int64(320

Quét lại toàn bộ 32 triệu dòng đánh giá theo từng mảnh nhỏ để lọc và giữ lại đúng lịch sử xem phim của 20 nghìn người dùng đã

In [ ]:
sampled_chunks = []
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open('ml-32m/ratings.csv') as f:
        reader = pd.read_csv(f, chunksize=chunk_size, dtype={'userId': np.int32, 'movieId': np.int32, 'rating': np.float32, 'timestamp': np.int64})
        for chunk in reader:
            chunk_filtered = chunk[chunk['userId'].isin(top_users_set)]
            if len(chunk_filtered) > 0:
                sampled_chunks.append(chunk_filtered)

In [ ]:
print(sampled_chunks)

[         userId  movieId  rating   timestamp
559          10        1     2.5  1169265231
560          10        2     2.0  1208464447
561          10       10     4.0  1169260936
562          10       19     3.0  1169264513
563          10       21     2.0  1169303086
...         ...      ...     ...         ...
1999458   12769   255335     4.5  1640684653
1999459   12769   255337     2.0  1640684607
1999460   12769   263007     4.0  1640683678
1999461   12769   263009     5.0  1640683660
1999462   12769   270698     5.0  1678363512

[932071 rows x 4 columns],          userId  movieId  rating   timestamp
2000698   12779      318     2.0  1654345095
2000699   12779      364     3.5  1666756439
2000700   12779      527     1.5  1654345098
2000701   12779     1196     2.0  1654345106
2000702   12779     1610     3.0  1654346197
...         ...      ...     ...         ...
3999995   25095     7419     4.5  1250536922
3999996   25095     7439     4.5  1250534266
3999997   25095     7802  

Chuyển mảng trên thành df

In [ ]:
df_sample = pd.concat(sampled_chunks, ignore_index=True)
df_sample.head()

,userId,movieId,rating,timestamp
0,10,1,2.5,1169265231
1,10,2,2.0,1208464447
2,10,10,4.0,1169260936
3,10,19,3.0,1169264513
4,10,21,2.0,1169303086


Tạo label nếu mà người dùng đánh giá >=4 thì là 1 , ngược lại thì là 0

In [ ]:
df_sample['target'] = (df_sample['rating'] >= 4.0).astype(np.int8)
df_sample.head(20)

,userId,movieId,rating,timestamp,target
0,10,1,2.5,1169265231,0
1,10,2,2.0,1208464447,0
2,10,10,4.0,1169260936,1
3,10,19,3.0,1169264513,0
4,10,21,2.0,1169303086,0
5,10,32,3.0,1258676251,0
6,10,34,2.0,1175614146,0
7,10,39,1.5,1208464497,0
8,10,44,3.0,1169264906,0
9,10,47,4.0,1169262246,1


Sắp xếp lại theo từng người dùng, và thời gian đánh giá mới nhất , mục đích của việc này là để lảya những bộ phim gần nhất để lấy làm test, đảm bào là mô hình sẽ học từ quá khứ và đoán tương lai

In [ ]:
df_sample = df_sample.sort_values(['userId', 'timestamp']).reset_index(drop=True)
df_sample.head(20)

,userId,movieId,rating,timestamp,target
0,10,48,3.5,1169260535,0
1,10,2006,2.0,1169260570,0
2,10,1954,3.5,1169260574,0
3,10,1909,2.5,1169260591,0
4,10,1690,2.0,1169260599,0
5,10,5218,4.0,1169260634,1
6,10,858,5.0,1169260817,1
7,10,733,4.5,1169260842,1
8,10,49272,3.5,1169260870,0
9,10,10,4.0,1169260936,1


# Temporal split: Lấy 80% đầu tiên là Train, 20% gần nhất là Test

In [ ]:
df_sample['rank_pct'] = df_sample.groupby('userId').cumcount() / df_sample.groupby('userId')['timestamp'].transform('count')
train_mask = df_sample['rank_pct'] < 0.8


In [ ]:
df_train = df_sample[train_mask].drop(columns=['rank_pct'])
df_train.head(10)

,userId,movieId,rating,timestamp,target
0,10,48,3.5,1169260535,0
1,10,2006,2.0,1169260570,0
2,10,1954,3.5,1169260574,0
3,10,1909,2.5,1169260591,0
4,10,1690,2.0,1169260599,0
5,10,5218,4.0,1169260634,1
6,10,858,5.0,1169260817,1
7,10,733,4.5,1169260842,1
8,10,49272,3.5,1169260870,0
9,10,10,4.0,1169260936,1


In [ ]:
df_test = df_sample[~train_mask].drop(columns=['rank_pct'])
df_test.head(10)

,userId,movieId,rating,timestamp,target
528,10,76251,4.0,1272714819,1
529,10,77561,3.0,1273642076,0
530,10,74789,2.5,1274608709,0
531,10,77866,2.0,1275644192,0
532,10,79139,2.5,1280754084,0
533,10,79132,4.0,1280846053,1
534,10,78105,3.0,1280846232,0
535,10,318,4.0,1280846249,1
536,10,253,3.5,1280846304,0
537,10,4226,3.5,1280870954,0


## 🔜 Kế Hoạch Các Giai Đoạn Tiếp Theo Trong Notebook

- **Giai đoạn 2 (Phase 2):** Xây dựng Module Collaborative Filtering (TruncatedSVD / Matrix Factorization) để lọc nhanh Top 300 ứng viên.
- **Giai đoạn 3 (Phase 3):** Xây dựng Mô hình Machine Learning (LightGBM Ranker) kết hợp đặc trưng phim, đặc trưng người dùng và CF dot product.
- **Giai đoạn 4 (Phase 4):** Pipeline kiểm thử End-to-End: Truyền vào `userId` bất kỳ $\rightarrow$ Trả về Top 10 phim gợi ý kèm xác suất % thích.